In [2]:
"""Evaluate NodePK model losses on the empirical Lenuzza dataset.

This utility mirrors the empirical batch loading performed by other scripts in
``scripts/models``.  It loads the default ``NodePK`` configuration and empirical
``StudyJSON`` file, runs a forward pass of :class:`~pff.models.amortized_inference.aicme.AICMEPK`
without gradient tracking, and reports the per-individual as well as
per-substance metrics produced by :class:`pff.models.amortized_inference.aicme.LossOutputs`.

Results are rendered as :mod:`pandas` tables on stdout, and optional CSV
exports can be requested via command line arguments.
"""


import argparse
from pathlib import Path
from typing import List, Sequence
import sys

import pandas as pd
import torch

from dataclasses import dataclass
from pathlib import Path


from pff import config_dir, data_dir  # noqa: E402
from pff.config_classes.node_pk_config import NodePKConfig  # noqa: E402
from pff.data_empirical import load_empirical_json_batches_as_dm  # noqa: E402
from pff.datasets.aicme_datasets import (  # noqa: E402
    AICMECompartmentsDataBatch,
    AICMECompartmentsDataModule,
)
from pff.models.amortized_inference.aicme import (  # noqa: E402
    AICMEPK,
    LossOutputs,
)


def _load_model(cfg: NodePKConfig, checkpoint: Path | None) -> AICMEPK:
    """Instantiate ``AICMEPK`` and optionally restore a checkpoint."""

    model = AICMEPK(cfg)
    if checkpoint is not None:
        state = torch.load(checkpoint, map_location="cpu")
        state_dict = state.get("state_dict", state)
        model.load_state_dict(state_dict)
    model.eval()
    return model


def _forward_loss(
    model: AICMEPK,
    batches: Sequence[AICMECompartmentsDataBatch],
) -> LossOutputs:
    """Run a forward pass on ``batches`` and collect ``LossOutputs``."""

    with torch.no_grad():
        loss_outputs, *_ = model(batches)
    return loss_outputs


def _per_individual_table(per_individual: List[dict[str, object]]) -> pd.DataFrame:
    """Convert nested per-individual metrics to a tidy :class:`DataFrame`."""

    if not per_individual:
        return pd.DataFrame()

    # ``pd.json_normalize`` flattens the ``metrics`` dictionary into columns.
    records = pd.json_normalize(per_individual)
    records.rename(columns=lambda col: col.replace("metrics.", ""), inplace=True)
    return records


def _per_substance_table(per_substance: dict[str, dict[str, float]]) -> pd.DataFrame:
    """Convert per-substance aggregates to a :class:`DataFrame`."""

    if not per_substance:
        return pd.DataFrame()

    df = pd.DataFrame.from_dict(per_substance, orient="index")
    df.index.name = "substance"
    df.reset_index(inplace=True)
    return df


@dataclass
class NodePKArgs:
    yaml: Path = Path(config_dir) / "experiment_configs" / "node-pk" / "base-homogeneous.yaml"
    json: Path = Path(data_dir) / "preprocessed" / "lenuzza_2016.json"
    checkpoint: Path | None = None
    per_individual_csv: Path | None = None
    per_substance_csv: Path | None = None

# Instead of parser.parse_args(), just instantiate:
args = NodePKArgs()

In [4]:
cfg: NodePKConfig = NodePKConfig.from_yaml(str(args.yaml))
datamodule = AICMECompartmentsDataModule(cfg)
datamodule.prepare_data()
datamodule.setup()

batches: List[AICMECompartmentsDataBatch] = load_empirical_json_batches_as_dm(
    args.json,
    meta_dosing=cfg.dosing,
    datamodule=datamodule,
)

In [11]:
batch = batches[1]

In [12]:
batch.context_subject_name

[['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '']]

In [6]:
loss_outputs

LossOutputs(agg_losses={'loss': tensor(1.7619), 'recon_loss': tensor(0.), 'kl_s': tensor(0.), 'kl_i': tensor(0.), 'kl_init': tensor(0.), 'kl_zs_zsN': tensor(0.), 'kl_zi_ziN': tensor(6.4716e-05), 'rmse': tensor(0.), 'log_rmse': tensor(0.), 'r2': tensor(0.), 'log_r2': tensor(0.), 'init_rmse': tensor(0.), 'pred_loss': tensor(0.8589), 'pred_rmse': tensor(0.0005), 'pred_log_rmse': tensor(2.0575), 'pred_r2': tensor(-2.1519), 'pred_log_r2': tensor(0.4069), 'invariance': tensor(0.9030)}, per_individual=[{'study': 'Lenuzza2016', 'substance': 'memantine', 'subject': '', 'metrics': {'rmse': 0.0, 'log_rmse': 0.0, 'r2': 1.0, 'log_r2': 1.0}}, {'study': 'Lenuzza2016', 'substance': 'omeprazole', 'subject': '', 'metrics': {'rmse': 0.00011178091517649591, 'log_rmse': 2.7691566944122314, 'r2': -2.7484920024871826, 'log_r2': -16.772457122802734}}, {'study': 'Lenuzza2016', 'substance': '5-hydroxyomeprazole', 'subject': '', 'metrics': {'rmse': 0.00011374797759344801, 'log_rmse': 3.4557931423187256, 'r2': -6

In [ ]:
print("Aggregated losses:")
agg_series = pd.Series({k: float(v) for k, v in loss_outputs.agg_losses.items()})
print(agg_series.to_string())
print()

per_individual_df = _per_individual_table(loss_outputs.per_individual)
if not per_individual_df.empty:
    per_individual_df.sort_values(["study", "substance", "subject"], inplace=True)
    print("Per-individual metrics:")
    print(per_individual_df.to_string(index=False))
else:
    print("Per-individual metrics: none available.")
print()

per_substance_df = _per_substance_table(loss_outputs.per_substance)
if not per_substance_df.empty:
    per_substance_df.sort_values("substance", inplace=True)
    print("Per-substance metrics:")
    print(per_substance_df.to_string(index=False))
else:
    print("Per-substance metrics: none available.")

if args.per_individual_csv is not None and not per_individual_df.empty:
    per_individual_df.to_csv(args.per_individual_csv, index=False)
    print(f"Saved per-individual metrics to {args.per_individual_csv}")

if args.per_substance_csv is not None and not per_substance_df.empty:
    per_substance_df.to_csv(args.per_substance_csv, index=False)
    print(f"Saved per-substance metrics to {args.per_substance_csv}")
